# AB Test

Para correr esta prueba necesitas cumplir con las siguientes precondiciones:

1.Entrena los modelos 

```
mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000

mlflow run . -e train --env-manager local --experiment-name mibici-equipo41 -P question=p3
mlflow run . -e train --env-manager local --experiment-name mibici-equipo41 -P question=p1 -P model=random_forest -P ... (model params)
```

2. Configura las variables de entorno para la API -> esto se ejemplifica en [README.md] -> ## Correr FastAPI con Docker -> ### Variables de entorno

3. Descarga el modelo del registry
```
 python -m mibici.api.scripts.export_model
```

4. Corre la API
```
uvicorn mibici.api.main:app --reload
```

5. Ejecuta el notebook


In [20]:
import pandas as pd
from mibici.config import STATIONS_DATA
from mibici.dataset import build_initial_dataset
from mibici.p1.dataset import create_p1_dataset
import requests
from mibici.config import PROJ_ROOT


#### Preparando los datos de octubre
Se ejecutará la prueba con una muestra de los datos de octubre

In [15]:
october = build_initial_dataset({"monthly": ["datos_abiertos_2026_10.csv"], "stations": STATIONS_DATA})
dataset = create_p1_dataset(october)
dataset.head()

,Origen_Id,Fecha,Viajes,Hora,Dia
0,272,2026-10-01 00:00:00,3,0,Thursday
1,272,2026-10-01 01:00:00,0,1,Thursday
2,272,2026-10-01 02:00:00,0,2,Thursday
3,272,2026-10-01 03:00:00,0,3,Thursday
4,272,2026-10-01 04:00:00,0,4,Thursday


In [16]:
dataset.drop(columns=['Fecha'], inplace=True)
sample = dataset.groupby(["Dia", "Hora"]).sample(n=60, random_state=42)
sample.shape

(10080, 4)

### Ejecutando prueba localmente

In [19]:

URL = "http://localhost:8000/predict/p1/abtest"
with requests.Session() as session:
    for row in sample.itertuples(index=False):
        session.post(URL, json={
            "origen_id": int(row.Origen_Id),
            "hora": int(row.Hora),
            "dia": row.Dia,
            "y": float(row.Viajes),
        }).raise_for_status()

### Resultados de la prueba
La api se encarga de registrar los resultados de la predicción en un csv para su consumo en este notebook. Se calculan dos métricas MAE y RMSE. Observamos que el modelo challenger, con un contexto más amplio de datos de entrenamiento, logra generalizar mejor para el mes de octubre por centesimas. 

In [22]:

log = pd.read_csv(PROJ_ROOT / "logs" / "p1_predictions.csv")
log["error"] = log["prediction"] - log["y"]
log.groupby("model").agg(
    requests=("y", "size"),
    mae=("error", lambda e: e.abs().mean()),
    rmse=("error", lambda e: (e ** 2).mean() ** 0.5),
)

,requests,mae,rmse
model,,,
challenger,5048,1.152981,2.255473
champion,5033,1.163853,2.302863
